In [13]:
%load_ext autoreload
%autoreload 2

In [1]:
from src.data_preprocessing import generate_datasets

generate_datasets("E4.data", [1, 64])

Procesando archivo: E4.data
Dataset guardado en: data/datasets/1-1.data
Dataset guardado en: data/datasets/2-64.data


In [1]:
from src.models.transformer import Transformer

# Parámetros del modelo
src_dim = 3      # features de los bloques
tgt_dim = 13     # features de las acciones
num_heads = 4
head_dim = 32
num_layers = 2

# Crear modelo
model = Transformer(
    src_dim=src_dim,
    tgt_dim=tgt_dim,
    num_heads=num_heads,
    head_dim=head_dim,
    num_layers=num_layers
)

In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy
from src.data_generator import load_data_from_file
from src.data_preprocessing import remove_elements_with_zero, remove_elements_with_less_blocks

def normalize_input(X):
    # Escalar con StandardScaler
    # X shape: [num_ejemplos, num_acciones, 4]
    X = np.array(X, dtype=np.float32)

    # Aplano a 2D
    X_flat = X.reshape(-1, X.shape[-1])  # [num_ejemplos*num_acciones, 4]

    # Fit/transform
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat)

    # Vuelvo a la forma original
    X = X_scaled.reshape(-1, X.shape[1], X.shape[2])
    return X

def curriculum_learning(model, datasets, epochs, weights, train_size, test_size, batch_size, learning_rate,
                        patience=5):
    """
    Entrena un modelo con curriculum learning y early stopping en función de val_loss.

    Args:
        model: Modelo de PyTorch.
        datasets: Lista de rutas a datasets.
        epochs: Lista con número de épocas por fase.
        weights: Lista con pesos de cada dataset.
        train_size: Número total de muestras de entrenamiento por fase.
        test_size: Número total de muestras de validación por fase.
        batch_size: Tamaño de batch.
        learning_rate: Tasa de aprendizaje.
        patience: Número de épocas sin mejora antes de detener.
    """
    # Cargar los datasets
    data = []
    for dataset in datasets:
        X, Y, blocks_ids = load_data_from_file(dataset)
        X, Y, blocks_ids = remove_elements_with_zero(X, Y, blocks_ids)
        X, Y, blocks_ids = remove_elements_with_less_blocks(X, Y, blocks_ids, 10000)
        X_enc, X_dec = zip(*X)
        X_dec = normalize_input(X_dec)

        X_enc = torch.tensor(X_enc, dtype=torch.float32)
        X_dec = torch.tensor(X_dec, dtype=torch.float32)
        Y = torch.tensor(Y, dtype=torch.float32)
        data.append((X_enc, X_dec, Y))

    # Inicializar el modelo, la función de pérdida y el optimizador
    loss_function = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Histórico de métricas
    history = {split: {'loss': [], 'acc': []} for split in ['train', 'val']}

    # Early stopping
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    no_improve_epochs = 0

    # Ciclo de entrenamiento por fases
    for phase_idx in range(len(datasets)):
        # Determinar la distribución de los datasets por fase
        total_weight = sum(weights[:phase_idx + 1])
        weights_normalized = [w / total_weight for w in weights[:phase_idx + 1]]

        train_loader_list, test_loader_list = [], []

        for idx, (X_enc, X_dec, Y) in enumerate(data[:phase_idx + 1]):
            weight = weights_normalized[idx]
            num_train_samples = int(weight * train_size)
            num_test_samples = int(weight * test_size)

            dataset = TensorDataset(X_enc, X_dec, Y)
            train_dataset, remaining_dataset = random_split(dataset, [num_train_samples, len(dataset) - num_train_samples])
            test_dataset, _ = random_split(remaining_dataset, [num_test_samples, len(remaining_dataset) - num_test_samples])

            train_loader_list.append(DataLoader(train_dataset, batch_size=batch_size, shuffle=True))
            test_loader_list.append(DataLoader(test_dataset, batch_size=batch_size, shuffle=False))

        # Ciclo de entrenamiento por épocas
        for epoch in range(epochs[phase_idx]):
            model.train()
            train_loss, correct, total = 0, 0, 0

            current_train_loader = []
            for loader in train_loader_list:
                current_train_loader.extend(loader)

            for X_enc_batch, X_dec_batch, y_batch in current_train_loader:
                optimizer.zero_grad()
                outputs = model(X_enc_batch, X_dec_batch)
                loss = loss_function(outputs, y_batch.argmax(dim=-1))
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * X_dec_batch.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch.argmax(dim=1)).sum().item()

            train_loss /= total
            train_accuracy = 100 * correct / total
            history['train']['loss'].append(train_loss)
            history['train']['acc'].append(train_accuracy)

            # Validación
            model.eval()
            validation_loss, correct, total = 0, 0, 0.01
            current_test_loader = []
            for loader in test_loader_list:
                current_test_loader.extend(loader)

            with torch.no_grad():
                for X_enc_batch, X_dec_batch, y_batch in current_test_loader:
                    outputs = model(X_enc_batch, X_dec_batch)
                    loss = loss_function(outputs, y_batch.argmax(dim=-1))
                    validation_loss += loss.item() * X_dec_batch.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    total += y_batch.size(0)
                    correct += (predicted == y_batch.argmax(dim=1)).sum().item()

            validation_loss /= total
            validation_accuracy = 100 * correct / total
            history['val']['loss'].append(validation_loss)
            history['val']['acc'].append(validation_accuracy)

            print(f'Epoch {epoch + 1}/{epochs[phase_idx]}, Phase {phase_idx + 1} - '
                  f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%')
            print(f'Epoch {epoch + 1}/{epochs[phase_idx]}, Phase {phase_idx + 1} - '
                  f'Val Loss: {validation_loss:.4f}, Val Accuracy: {validation_accuracy:.2f}%')

            # --- EARLY STOPPING SOLO EN LOSS ---
            if validation_loss < best_loss:
                best_loss = validation_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1
                if no_improve_epochs >= patience:
                    print(f"Early stopping en la epoch {epoch+1} de la fase {phase_idx+1}")
                    model.load_state_dict(best_model_wts)
                    return history, model

    # Al terminar todas las fases, restauramos el mejor modelo
    model.load_state_dict(best_model_wts)
    return history, model

In [ ]:
datasets = ["datasets/1-1.data", "datasets/2-64.data"]
epochs = [0, 3]
weights = [50, 50]
train_size = 100
test_size = 100
batch_size = 32
learning_rate = 1e-4
patience=5 # early stopping

curriculum_learning(model, datasets, epochs, weights, train_size, test_size, batch_size, learning_rate, patience)

FileNotFoundError: [Errno 2] No such file or directory: 'data/datasets/2-64.data'

# Decoder-only model

In [4]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split, Subset
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy
from src.data_generator import load_data_from_file
from src.data_preprocessing import remove_elements_with_zero, remove_elements_with_less_blocks
import random
import gc

def normalize_input(X):
    # Escalar con StandardScaler
    # X shape: [num_ejemplos, num_acciones, 4]
    X = np.array(X, dtype=np.float32)

    # Aplano a 2D
    X_flat = X.reshape(-1, X.shape[-1])  # [num_ejemplos*num_acciones, 4]

    # Fit/transform
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_flat)

    # Vuelvo a la forma original
    X = X_scaled.reshape(-1, X.shape[1], X.shape[2])
    return X


def expand_dataset(dataset, target_size):
    """Duplica muestras aleatorias del dataset hasta alcanzar target_size."""
    current_size = len(dataset)
    if current_size >= target_size:
        return Subset(dataset, random.sample(range(current_size), target_size))

    # Calcula cuántas muestras adicionales se necesitan
    extra_needed = target_size - current_size

    # Selecciona índices aleatorios para duplicar
    extra_indices = random.choices(range(current_size), k=extra_needed)

    # Combina los índices originales con los duplicados
    all_indices = list(range(current_size)) + extra_indices
    random.shuffle(all_indices)

    return Subset(dataset, all_indices)


def curriculum_learning(model, datasets, epochs, weights, train_size, test_size, batch_size, learning_rate,
                        patience=5):
    """
    Entrena un modelo con curriculum learning y early stopping en función de val_loss.

    Args:
        model: Modelo de PyTorch.
        datasets: Lista de rutas a datasets.
        epochs: Lista con número de épocas por fase.
        weights: Lista con pesos de cada dataset.
        train_size: Número total de muestras de entrenamiento por fase.
        test_size: Número total de muestras de validación por fase.
        batch_size: Tamaño de batch.
        learning_rate: Tasa de aprendizaje.
        patience: Número de épocas sin mejora antes de detener.
    """
    # Cargar los datasets
    data = []
    for dataset in datasets:
        X_src, X_tgt, Y, blocks_ids = load_data_from_file(dataset)
        X_src, X_tgt, Y, blocks_ids = remove_elements_with_zero(X_src, X_tgt, Y, blocks_ids)
        X_tgt = normalize_input(X_tgt)
        X_tgt = torch.tensor(X_tgt, dtype=torch.float32)
        Y = torch.tensor(Y, dtype=torch.float32)
        data.append((X_tgt, Y))

        del X_src, blocks_ids, X_tgt, Y
        gc.collect()

    # Inicializar el modelo, la función de pérdida y el optimizador
    loss_function = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Histórico de métricas
    history = {split: {'loss': [], 'acc': []} for split in ['train', 'val']}

    # Early stopping
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    no_improve_epochs = 0

    # Ciclo de entrenamiento por fases
    for phase_idx in range(len(datasets)):
        # Determinar la distribución de los datasets por fase
        total_weight = sum(weights[:phase_idx + 1])
        weights_normalized = [w / total_weight for w in weights[:phase_idx + 1]]

        train_loader_list, test_loader_list = [], []

        for idx, (X_tgt, Y) in enumerate(data[:phase_idx + 1]):
            weight = weights_normalized[idx]
            num_train_samples = int(weight * train_size)
            num_test_samples = int(weight * test_size)

            dataset = TensorDataset(X_tgt, Y)

            # --- PRIMERO: crear el set de validación (sin duplicar)
            num_test_actual = min(num_test_samples, len(dataset))
            test_dataset, remaining_dataset = random_split(
                dataset,
                [num_test_actual, len(dataset) - num_test_actual]
            )

            # --- LUEGO: crear el set de entrenamiento a partir de los datos restantes
            num_train_actual = min(num_train_samples, len(remaining_dataset))
            train_dataset, _ = random_split(
                remaining_dataset,
                [num_train_actual, len(remaining_dataset) - num_train_actual]
            )

            # --- EXPANDIR SOLO EL SET DE ENTRENAMIENTO
            train_dataset = expand_dataset(train_dataset, num_train_samples)

            train_loader_list.append(DataLoader(train_dataset, batch_size=batch_size, shuffle=True))
            test_loader_list.append(DataLoader(test_dataset, batch_size=batch_size, shuffle=False))

        # Ciclo de entrenamiento por épocas
        for epoch in range(epochs[phase_idx]):
            model.train()
            train_loss, correct, total = 0, 0, 0

            current_train_loader = []
            for loader in train_loader_list:
                current_train_loader.extend(loader)

            for X_tgt_batch, y_batch in current_train_loader:
                optimizer.zero_grad()
                outputs = model(X_tgt_batch)
                loss = loss_function(outputs, y_batch.argmax(dim=-1))
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * X_tgt_batch.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += y_batch.size(0)
                correct += (predicted == y_batch.argmax(dim=1)).sum().item()

            train_loss /= total
            train_accuracy = 100 * correct / total
            history['train']['loss'].append(train_loss)
            history['train']['acc'].append(train_accuracy)

            # Validación
            model.eval()
            validation_loss, correct, total = 0, 0, 0.01
            current_test_loader = []
            for loader in test_loader_list:
                current_test_loader.extend(loader)

            with torch.no_grad():
                for X_tgt_batch, y_batch in current_test_loader:
                    outputs = model(X_tgt_batch)
                    loss = loss_function(outputs, y_batch.argmax(dim=-1))
                    validation_loss += loss.item() * X_tgt_batch.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    total += y_batch.size(0)
                    correct += (predicted == y_batch.argmax(dim=1)).sum().item()

            validation_loss /= total
            validation_accuracy = 100 * correct / total
            history['val']['loss'].append(validation_loss)
            history['val']['acc'].append(validation_accuracy)

            print(f'Epoch {epoch + 1}/{epochs[phase_idx]}, Phase {phase_idx + 1} - '
                  f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%')
            print(f'Epoch {epoch + 1}/{epochs[phase_idx]}, Phase {phase_idx + 1} - '
                  f'Val Loss: {validation_loss:.4f}, Val Accuracy: {validation_accuracy:.2f}%')

            # --- EARLY STOPPING SOLO EN LOSS ---
            if validation_loss < best_loss:
                best_loss = validation_loss
                best_model_wts = copy.deepcopy(model.state_dict())
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1
                if no_improve_epochs >= patience:
                    print(f"Early stopping en la epoch {epoch+1} de la fase {phase_idx+1}")
                    model.load_state_dict(best_model_wts)
                    return history, model

    # Al terminar todas las fases, restauramos el mejor modelo
    model.load_state_dict(best_model_wts)
    return history, model

In [8]:
from src.models.decoder_model import DecoderModel

# Parámetros del modelo
input_dim = 4     # features de las acciones
num_heads = 4
head_dim = 32
num_layers = 4

# Crear modelo
model = DecoderModel(
    input_dim=input_dim,
    num_heads=num_heads,
    head_dim=head_dim,
    num_layers=num_layers,
    dropout_rate=0.1
)

In [9]:
datasets = ["datasets/E4C_2-8.data", "datasets/E4C_1-1.data"]
epochs = [0, 50]
weights = [50, 50]
train_size = 4000
test_size = 600
batch_size = 32
learning_rate = 1e-4
patience=5 # early stopping

_, model = curriculum_learning(model, datasets, epochs, weights, train_size, test_size, batch_size, learning_rate, patience)

Epoch 1/50, Phase 2 - Train Loss: 1.9788, Train Accuracy: 40.95%
Epoch 1/50, Phase 2 - Val Loss: 2.8863, Val Accuracy: 46.33%
Epoch 2/50, Phase 2 - Train Loss: 1.8469, Train Accuracy: 42.42%
Epoch 2/50, Phase 2 - Val Loss: 3.1776, Val Accuracy: 48.50%
Epoch 3/50, Phase 2 - Train Loss: 1.8513, Train Accuracy: 42.40%
Epoch 3/50, Phase 2 - Val Loss: 3.4626, Val Accuracy: 47.83%
Epoch 4/50, Phase 2 - Train Loss: 1.8799, Train Accuracy: 44.17%
Epoch 4/50, Phase 2 - Val Loss: 3.5054, Val Accuracy: 47.83%
Epoch 5/50, Phase 2 - Train Loss: 1.8879, Train Accuracy: 44.00%
Epoch 5/50, Phase 2 - Val Loss: 3.2250, Val Accuracy: 47.17%
Epoch 6/50, Phase 2 - Train Loss: 1.8264, Train Accuracy: 44.55%
Epoch 6/50, Phase 2 - Val Loss: 3.3894, Val Accuracy: 47.33%
Early stopping en la epoch 6 de la fase 2


In [26]:
from src.training import *

save_model(model, "decoder.pth")

In [7]:
gc.collect()

3198